# Pilot Model v5 — weighted tuned XGBoost

**Pilot Model v5** фиксирует текущий лучший standalone Nov–Dec leaderboard pipeline: tuned XGBoost + `SET_FEATURES + CALENDAR_FEATURES + WEATHER_FEATURES` (C+W) + weighted D. Сообщённый leaderboard score — **0.88663**. Это воспроизведение выбранной конфигурации, без нового HPO, feature selection или перебора весов.

Единица прогноза — `(route, date, hour)`, target — `boardings`. Jan–Oct 2025 содержит все 72 960 известных target-строк и используется для final fit. Nov–Dec 2025 содержит ровно предоставленную forecast grid из 14 640 строк без target: эти даты не используются для настройки параметров, признаков или half-life. Исторические temporal folds уже использовались в model selection и HPO; здесь validation ничего не выбирает.

## Environment & Data Setup

Загружаем напрямую готовые model-ready CSV. Labels, grid и features из raw заново не строятся. Источники HPO и model selection читаются только для проверки зафиксированного контракта.

In [1]:
from pathlib import Path
from time import perf_counter
import json
import sys
import numpy as np
import pandas as pd
from IPython.display import display
from xgboost import XGBRegressor

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'ml/src/model_selection.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Project root not found')
sys.path.insert(0, str(ROOT / 'ml/src'))
import model_selection as ms

DATA_ROOT = ROOT / 'data'
TRAIN_PATH = DATA_ROOT / 'processed/model_ready_train_2025_jan_oct.csv'
FORECAST_PATH = DATA_ROOT / 'processed/model_ready_forecast_2025_nov_dec.csv'
MODEL_SELECTION_PATH = ROOT / 'ml/experiments/results/model_selection.json'
HPO_PATH = ROOT / 'ml/experiments/results/hyperparameter_tuning.json'
SUBMISSION_PATH = DATA_ROOT / 'submissions/pilot_model_v5_weighted_xgboost_submission.csv'
BUCKET_DTYPE = ms.BUCKET_DTYPE

train = pd.read_csv(TRAIN_PATH, sep=';', encoding='utf-8', parse_dates=['date'],
                    dtype={'temperature_bucket': BUCKET_DTYPE})
forecast = pd.read_csv(FORECAST_PATH, sep=';', encoding='utf-8', parse_dates=['date'],
                       dtype={'temperature_bucket': BUCKET_DTYPE})
model_selection = json.loads(MODEL_SELECTION_PATH.read_text(encoding='utf-8'))
hpo = json.loads(HPO_PATH.read_text(encoding='utf-8'))
display(pd.DataFrame([{'train_rows': len(train), 'forecast_rows': len(forecast),
                       'train_last_date': train.date.max().date(),
                       'forecast_first_date': forecast.date.min().date(),
                       'forecast_last_date': forecast.date.max().date()}]))

,train_rows,forecast_rows,train_last_date,forecast_first_date,forecast_last_date
0,72960,14640,2025-10-31,2025-11-01,2025-12-31


## Data Contract / Sanity Checks

До обучения проверяем schema, полноту route/date/hour grid, целевой столбец только в train, календарные и категориальные значения. Route 5 присутствует в обеих grid и остаётся в submission.

In [2]:
ROUTES = ms.ROUTES
KEYS = ['route', 'date', 'hour']
SET_FEATURES = ms.SET_FEATURES
CALENDAR_FEATURES = ms.CALENDAR_FEATURES
WEATHER_FEATURES = ms.WEATHER_FEATURES
CALENDAR_WEATHER_FEATURES = SET_FEATURES + CALENDAR_FEATURES + WEATHER_FEATURES
BUCKET_LABELS = tuple(BUCKET_DTYPE.categories)

WEATHER_COLUMNS = (
    'climatological_temperature', 'climatological_precipitation',
    'climatological_rain_probability', 'temperature_bucket'
)
BUCKET_LABELS = ('<0', '[0,10)', '[10,20)', '>=20')
FORECAST_COLUMNS = [
    'route', 'date', 'hour', 'weekday', 'route_hour', 'hour_weekend',
    'is_night', *CALENDAR_FEATURES, *WEATHER_COLUMNS
]
TRAIN_COLUMNS = FORECAST_COLUMNS[:-len(WEATHER_COLUMNS)] + ['boardings', *WEATHER_COLUMNS]

if train.columns.tolist() != TRAIN_COLUMNS:
    raise ValueError('Train schema differs from model-ready contract')
if forecast.columns.tolist() != FORECAST_COLUMNS:
    raise ValueError('Forecast schema differs from model-ready contract')
if train.drop(columns='boardings').columns.tolist() != forecast.columns.tolist():
    raise ValueError('Train and forecast feature columns differ')
if set(train.columns) - set(KEYS) - {'boardings'} != set(SET_FEATURES + CALENDAR_FEATURES + WEATHER_COLUMNS) - set(KEYS):
    raise ValueError('Unexpected feature columns')

def check_grid(frame, start, end, expected_rows):
    dates = pd.date_range(start, end, freq='D')
    if len(frame) != expected_rows or frame[KEYS].isna().any().any():
        raise ValueError('Incorrect row count or missing keys')
    if frame.isna().any().any():
        raise ValueError('Missing values in model-ready data')
    if not pd.api.types.is_datetime64_any_dtype(frame['date']):
        raise TypeError('date must be datetime')
    if frame[KEYS].duplicated().any():
        raise ValueError('Duplicate route/date/hour keys')
    if set(frame['route']) != set(ROUTES) or set(frame['hour']) != set(range(24)):
        raise ValueError('Unexpected route or hour values')
    if set(frame['date']) != set(dates):
        raise ValueError('Incomplete or unexpected date period')
    if len(frame) != len(ROUTES) * len(dates) * 24:
        raise ValueError('Incomplete route/date/hour grid')
    if not all(pd.api.types.is_numeric_dtype(frame[col]) for col in frame.columns if col not in ('date', 'temperature_bucket')):
        raise TypeError('Non-numeric feature or target')
    if frame['temperature_bucket'].dtype != BUCKET_DTYPE:
        raise TypeError('Temperature bucket categorical dtype changed')
    if not frame['temperature_bucket'].isin(BUCKET_LABELS).all():
        raise ValueError('Unknown temperature bucket')
    expected_bucket = pd.cut(
        frame['climatological_temperature'], bins=[-np.inf, 0, 10, 20, np.inf],
        right=False, labels=BUCKET_LABELS,
    )
    if not frame['temperature_bucket'].eq(expected_bucket).all():
        raise ValueError('Temperature bucket differs from frozen climatology')
    if not frame['weekday'].eq(frame['date'].dt.dayofweek).all():
        raise ValueError('weekday does not match date')
    if not frame['route_hour'].eq(frame['route'] * 24 + frame['hour']).all():
        raise ValueError('route_hour encoding differs from v1')
    if not frame['hour_weekend'].eq(frame['hour'] * 2 + frame['weekday'].ge(5).astype('int8')).all():
        raise ValueError('hour_weekend encoding differs from v1')
    if not frame['is_night'].eq(frame['hour'].between(0, 5).astype('int8')).all():
        raise ValueError('is_night encoding differs from v1')
    for col in ('is_night', *CALENDAR_FEATURES):
        if not frame[col].isin((0, 1)).all():
            raise ValueError(f'{col} must be binary')

check_grid(train, '2025-01-01', '2025-10-31', 72_960)
check_grid(forecast, '2025-11-01', '2025-12-31', 14_640)
if 'boardings' in forecast.columns:
    raise ValueError('Forecast must not contain target')
if train['boardings'].lt(0).any():
    raise ValueError('Negative training target')
print('Model-ready data contract passed.')
display(pd.DataFrame([{'check': 'model-ready contract', 'status': 'passed',
                       'train_rows': len(train), 'forecast_rows': len(forecast),
                       'routes': forecast.route.nunique(), 'hours': forecast.hour.nunique()}]))

Model-ready data contract passed.


,check,status,train_rows,forecast_rows,routes,hours
0,model-ready contract,passed,72960,14640,10,24


## Feature Configuration & HPO Checkpoint

Модель получает ровно 12 C+W признаков: frozen core `SET_FEATURES`, пять `CALENDAR_FEATURES` и `WEATHER_FEATURES = ("temperature_bucket",)`. Остальные climatological weather columns хранятся в CSV для контекста, но не подаются в модель. Категориальные признаки обрабатывает `model_selection.make_features`: фиксированные pandas categories и native categorical splits XGBoost.

Tuned XGBoost выбран по текущему лучшему standalone Nov–Dec leaderboard score **0.88663** при прежнем HPO contract. Историческая HPO pooled WAPE для XGBoost — `0.153481`; leaderboard score сам по себе не является новой validation metric и здесь ничего не подбирает. Альтернатива — tuned LightGBM, но его weighted standalone leaderboard score ниже (`0.88431`). Проверяем точное совпадение features, preprocessing и параметров с сохранённым HPO.

In [3]:
ROUTES = ms.ROUTES
KEYS = ['route', 'date', 'hour']
SET_FEATURES = ms.SET_FEATURES
CALENDAR_FEATURES = ms.CALENDAR_FEATURES
WEATHER_FEATURES = ms.WEATHER_FEATURES
FEATURES = SET_FEATURES + CALENDAR_FEATURES + WEATHER_FEATURES
CONFIGURATION = 'calendar_weather'
XGBOOST_PARAMS = {**ms.XGBOOST_PARAMETERS, 'n_estimators': 500, 'max_depth': 6,
                  'learning_rate': 0.1, 'min_child_weight': 1}
if WEATHER_FEATURES != ('temperature_bucket',):
    raise ValueError('Weather feature contract changed')
if FEATURES != ms.FEATURE_CONFIGURATIONS[CONFIGURATION]:
    raise ValueError('C+W feature list changed')
if model_selection['features'][CONFIGURATION] != list(FEATURES):
    raise ValueError('Model-selection features changed')
if hpo['configuration'] != CONFIGURATION or hpo['features'] != list(FEATURES):
    raise ValueError('HPO feature contract changed')
if hpo['ensemble_candidate']['family'] != 'XGBRegressor' or hpo['ensemble_candidate']['parameters'] != XGBOOST_PARAMS:
    raise ValueError('Tuned XGBoost parameters changed')
if hpo['nov_dec_used'] or hpo['accepted_feature_registry_changed']:
    raise ValueError('HPO provenance contract changed')
for path in (TRAIN_PATH, MODEL_SELECTION_PATH, ROOT / 'ml/src/model_selection.py',
             ROOT / 'ml/src/hyperparameter_tuning.py'):
    relative = path.relative_to(ROOT).as_posix()
    if ms.sha256(path) != hpo['source_sha256'][relative]:
        raise ValueError(f'Source hash changed: {relative}')
x_train = ms.make_features(train, CONFIGURATION, 'XGBRegressor')
x_forecast = ms.make_features(forecast, CONFIGURATION, 'XGBRegressor')
if x_train.columns.tolist() != list(FEATURES) or not x_train.dtypes.equals(x_forecast.dtypes):
    raise ValueError('Preprocessing contract changed')
display(pd.DataFrame([{'family': 'XGBRegressor', 'features': len(FEATURES),
                       'n_estimators': XGBOOST_PARAMS['n_estimators'],
                       'max_depth': XGBOOST_PARAMS['max_depth'],
                       'learning_rate': XGBOOST_PARAMS['learning_rate'],
                       'min_child_weight': XGBOOST_PARAMS['min_child_weight'],
                       'seed': XGBOOST_PARAMS['random_state']}]))

,family,features,n_estimators,max_depth,learning_rate,min_child_weight,seed
0,XGBRegressor,12,500,6,0.1,1,42


## Weighted D / Final Training Jan–Oct

Weighted D задаёт `weight = exp(-ln(2) * age_days / 180)`, где `age_days = (latest_date - date).days`, а `latest_date = train["date"].max()` для **этого** fit. Half-life равен 180 дням: более свежие строки получают больший вес. Для final Jan–Oct fit origin — `2025-10-31`; веса historical folds не переиспользуются.

Historical validation weighting для LightGBM имел статус `RECENCY_WEIGHTING_REJECTED`; weighted D у XGBoost также уступил unweighted historical reference. Однако на Nov–Dec leaderboard weighted D улучшил standalone score обеих model families. Это эмпирическое основание зафиксировать текущий submission, без утверждения об универсальной пользе weighting. Обучаем один XGBoost на **всех** Jan–Oct строках; Nov–Dec target недоступен и не участвует в fit.

In [4]:
latest_date = train['date'].max()
age_days = (latest_date - train['date']).dt.days.to_numpy()
sample_weight = np.exp(-np.log(2) * age_days / 180)
if not np.isfinite(sample_weight).all() or (sample_weight <= 0).any() or sample_weight.max() != 1:
    raise ValueError('Invalid final training weights')
weight_diagnostic = train[['date']].copy()
weight_diagnostic['weight'] = sample_weight
weight_diagnostic['month'] = weight_diagnostic['date'].dt.strftime('%Y-%m')
display(pd.DataFrame([{'weight_origin': latest_date.date(), 'half_life_days': 180,
                       'min_weight': sample_weight.min(), 'max_weight': sample_weight.max(),
                       'effective_weight_sum': sample_weight.sum()}]).round(4))
display(weight_diagnostic.groupby('month', sort=True)['weight']
        .agg(rows='size', min_weight='min', mean_weight='mean',
             max_weight='max', effective_weight_sum='sum').round(4))

model = XGBRegressor(**XGBOOST_PARAMS)
start = perf_counter()
model.fit(x_train, train['boardings'], sample_weight=sample_weight)
training_seconds = perf_counter() - start
display(pd.DataFrame([{'trained_rows': len(train), 'fit_seconds': training_seconds}]).round(3))

,weight_origin,half_life_days,min_weight,max_weight,effective_weight_sum
0,2025-10-31,180,0.3114,1.0,43076.3592


,rows,min_weight,mean_weight,max_weight,effective_weight_sum
month,,,,,
2025-01,7440,0.3114,0.3301,0.3495,2455.7407
2025-02,6720,0.3508,0.3697,0.3893,2484.6593
2025-03,7440,0.3908,0.4143,0.4386,3082.1478
2025-04,7200,0.4403,0.4659,0.4924,3354.3202
2025-05,7440,0.4943,0.5240,0.5548,3898.2454
2025-06,7200,0.5569,0.5892,0.6227,4242.4842
2025-07,7440,0.6251,0.6627,0.7017,4930.4312
2025-08,7440,0.7044,0.7467,0.7907,5555.5742
2025-09,7200,0.7937,0.8397,0.8875,6046.1652


,trained_rows,fit_seconds
0,72960,16.96


## Nov–Dec Inference

Прогнозируем ровно строки предоставленного forecast CSV в исходном порядке. Postprocessing сохраняет v4 contract: `floor(max(raw, 0) + 0.5)` даёт неотрицательное целое число, затем для route 5 prediction принудительно равен `0`. Отсутствие NaN/inf проверяется до и после обработки.

In [5]:
start = perf_counter()
raw_prediction = model.predict(x_forecast)
prediction_seconds = perf_counter() - start
if not np.isfinite(raw_prediction).all():
    raise ValueError('Non-finite raw predictions')
prediction = np.floor(np.maximum(raw_prediction, 0) + 0.5).astype('int64')
prediction[forecast['route'].to_numpy() == 5] = 0
inference = forecast.loc[:, KEYS].copy()
inference['prediction'] = prediction
if len(inference) != 14_640 or inference.isna().any().any():
    raise ValueError('Incomplete inference')
if not np.isfinite(inference['prediction']).all() or inference['prediction'].lt(0).any():
    raise ValueError('Invalid predictions')
if not pd.api.types.is_integer_dtype(inference['prediction']):
    raise TypeError('Predictions are not integers')
if not inference.loc[inference.route.eq(5), 'prediction'].eq(0).all():
    raise ValueError('Route 5 policy failed')
display(pd.DataFrame([{'prediction_rows': len(inference), 'predict_seconds': prediction_seconds,
                       'route_5_nonzero': int(inference.loc[inference.route.eq(5), 'prediction'].ne(0).sum())}]).round(3))

,prediction_rows,predict_seconds,route_5_nonzero
0,14640,0.305,0


## Nov–Dec Inference Diagnostics

Эти агрегаты помогают проверить масштаб и распределение прогноза; без Nov–Dec target они не измеряют accuracy. Weekday/weekend определяется по календарному `weekday` forecast grid.

In [6]:
diagnostic = forecast.loc[:, KEYS + ['weekday']].copy()
diagnostic['prediction'] = prediction
diagnostic['month'] = diagnostic['date'].dt.month.map({11: 'Nov', 12: 'Dec'})
diagnostic['day_type'] = np.where(diagnostic['weekday'].lt(5), 'weekday', 'weekend')
values = diagnostic['prediction']
display(pd.DataFrame([{'total_predicted_boardings': int(values.sum()),
                       'median': float(values.median()), 'p95': float(values.quantile(0.95)),
                       'p99': float(values.quantile(0.99)), 'max': int(values.max())}]).round(2))
display(diagnostic.groupby('month', sort=False)['prediction'].sum().rename('total'))
display(diagnostic.groupby('day_type', sort=False)['prediction'].sum().rename('total'))
display(diagnostic.groupby('route', sort=True)['prediction'].sum().rename('total'))

,total_predicted_boardings,median,p95,p99,max
0,12602030,501.0,2961.3,3953.05,5316


month
Nov    6121622
Dec    6480408
Name: total, dtype: int64

day_type
weekend     2526771
weekday    10075259
Name: total, dtype: int64

route
1     1084915
5           0
7     1396519
11    1859540
12    1943065
17    2989994
25     417962
26    1042174
28     596379
50    1271482
Name: total, dtype: int64

## Submission

Сохраняем один основной Pilot v5 artifact: UTF-8, разделитель `;`, schema `route;date;hour;prediction`. Grid и порядок берутся непосредственно из forecast input; ensemble не создаётся.

In [7]:
submission = inference.loc[:, ['route', 'date', 'hour', 'prediction']].copy()
submission['date'] = submission['date'].dt.strftime('%Y-%m-%d')
submission.to_csv(SUBMISSION_PATH, sep=';', index=False, encoding='utf-8')
print('Saved:', SUBMISSION_PATH.relative_to(ROOT).as_posix(), 'rows:', len(submission))

Saved: data/submissions/pilot_model_v5_weighted_xgboost_submission.csv rows: 14640


## Final Sanity Checks

После записи повторно читаем CSV и сверяем schema, 14 640 строк, уникальность ключей, период, 10 routes, 24 hours и точный порядок ключей с forecast и organizer template. Проверяем отсутствие NaN/inf, неотрицательные целые predictions и полностью нулевой route 5.

In [8]:
expected_keys = forecast[KEYS].copy()
expected_keys['date'] = expected_keys['date'].dt.strftime('%Y-%m-%d')
expected_keys = expected_keys.reset_index(drop=True)
template = pd.read_csv(DATA_ROOT / 'raw/test_submission.csv', sep=';', encoding='utf-8')
if template[KEYS].isna().any().any() or not template[KEYS].equals(expected_keys):
    raise ValueError('Forecast keys/order differ from organizer template')
saved = pd.read_csv(SUBMISSION_PATH, sep=';', encoding='utf-8')
if saved.columns.tolist() != ['route', 'date', 'hour', 'prediction']:
    raise ValueError('Saved submission schema differs')
if len(saved) != 14_640 or saved[KEYS].duplicated().any():
    raise ValueError('Saved row count or unique keys differ')
if not saved[KEYS].equals(expected_keys):
    raise ValueError('Saved keys/order differ from forecast/template')
if set(saved['date']) != set(pd.date_range('2025-11-01', '2025-12-31').strftime('%Y-%m-%d')):
    raise ValueError('Saved period differs')
if set(saved['route']) != set(ROUTES) or len(ROUTES) != 10 or set(saved['hour']) != set(range(24)):
    raise ValueError('Saved routes or hours differ')
if saved.isna().any().any() or not np.isfinite(saved['prediction']).all():
    raise ValueError('Saved submission contains NaN/inf')
if saved['prediction'].lt(0).any() or not pd.api.types.is_integer_dtype(saved['prediction']):
    raise ValueError('Saved predictions must be non-negative integers')
if not saved.loc[saved['route'].eq(5), 'prediction'].eq(0).all():
    raise ValueError('Saved route 5 predictions must be zero')
if not saved.equals(submission.reset_index(drop=True)):
    raise ValueError('Saved submission differs from in-memory predictions')
display(pd.DataFrame([{'status': 'passed', 'rows': len(saved), 'routes': saved.route.nunique(),
                       'hours': saved.hour.nunique(), 'unique_keys': len(saved[KEYS].drop_duplicates()),
                       'route_5_nonzero': 0}]))

,status,rows,routes,hours,unique_keys,route_5_nonzero
0,passed,14640,10,24,14640,0


## Итог Pilot v5

Зафиксирован один standalone submission tuned XGBoost + C+W + weighted D (half-life 180 дней), обученный на всех Jan–Oct target-строках. Историческая классификация weighting и accepted feature registry не меняются. Сообщённый лучший standalone leaderboard score — `0.88663`; готовый ensemble 75/25 получил `0.88657` и не заменяет Pilot v5.